Imports

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

from imblearn.over_sampling import SMOTE

Load Data

## Carga de Datos

Se carga el conjunto de datos Telco Customer Churn, que contiene información demográfica, servicios contratados y estado de abandono de clientes.

El objetivo es construir modelos capaces de predecir qué clientes tienen mayor probabilidad de abandonar la compañía.

In [88]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [89]:
df.shape

(7043, 21)

In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Data Cleaning

## Limpieza de Datos

Se verificó la existencia de valores nulos y registros duplicados.

La variable TotalCharges fue convertida de tipo texto a formato numérico para permitir su utilización en los modelos de Machine Learning.

Posteriormente se eliminaron los registros con valores faltantes y se descartó la columna CustomerID por no aportar valor predictivo.

In [91]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [92]:
df.duplicated().sum()

np.int64(0)

In [93]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [94]:
df["TotalCharges"].isnull().sum()

np.int64(11)

In [95]:
df = df.dropna()

In [96]:
df = df.drop("customerID", axis=1)

Feature Engineering

## Ingeniería de Características

Las variables categóricas fueron transformadas a formato numérico para que pudieran ser interpretadas por los algoritmos de Machine Learning.

Las variables binarias se codificaron utilizando valores 0 y 1, mientras que las variables con múltiples categorías fueron transformadas mediante One-Hot Encoding.

In [97]:
binary_columns = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]

for col in binary_columns:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [98]:
df["gender"] = df["gender"].map({"Male": 1, "Female": 0})

In [99]:
categorical_cols = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [100]:
df_encoded.shape

(7032, 31)

X / y

## Variables Predictoras y Variable Objetivo

Se separó la variable objetivo Churn del resto de variables predictoras.

La variable Churn representa si un cliente abandonó o no la compañía.

In [101]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

In [102]:
print(X.shape)
print(y.shape)

(7032, 30)
(7032,)


Train Test Split

## División de Datos

El conjunto de datos fue dividido en entrenamiento (80%) y prueba (20%).

Los datos de entrenamiento se utilizan para construir los modelos, mientras que los datos de prueba permiten evaluar su rendimiento sobre información no vista previamente.

In [103]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Scaling

## Escalado de Variables

Se aplicó StandardScaler para estandarizar las variables numéricas.

Este proceso ayuda a que algunos algoritmos, como KNN y Regresión Logística, funcionen de manera más eficiente y estable.

In [104]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Baseline Models

## Modelos Base

Se entrenaron modelos iniciales utilizando los datos originales para obtener una referencia de rendimiento antes de aplicar técnicas de balanceo de clases.

In [105]:
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

print(classification_report(y_test, knn_pred))

              precision    recall  f1-score   support

           0       0.83      0.84      0.83      1033
           1       0.54      0.51      0.52       374

    accuracy                           0.75      1407
   macro avg       0.68      0.67      0.68      1407
weighted avg       0.75      0.75      0.75      1407



In [106]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

log_pred = log_reg.predict(X_test_scaled)

print(classification_report(y_test, log_pred))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407



Imbalanced Data — SMOTE

## Balanceo de Clases con SMOTE

El conjunto de datos presenta un desbalance entre clientes que abandonan y clientes que permanecen.

Para reducir este problema se utilizó SMOTE, una técnica que genera ejemplos sintéticos de la clase minoritaria con el objetivo de equilibrar la distribución de clases.

In [107]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

In [108]:
y_train_smote.value_counts()

Churn
1    4130
0    4130
Name: count, dtype: int64

In [109]:
log_reg_smote = LogisticRegression(max_iter=1000)
log_reg_smote.fit(X_train_smote, y_train_smote)

smote_pred = log_reg_smote.predict(X_test_scaled)

print(classification_report(y_test, smote_pred))

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1033
           1       0.50      0.80      0.62       374

    accuracy                           0.74      1407
   macro avg       0.70      0.76      0.71      1407
weighted avg       0.80      0.74      0.75      1407



Models

## Comparación de Modelos

Se entrenaron distintos algoritmos de Machine Learning utilizando los datos balanceados con SMOTE.

El objetivo es comparar su capacidad predictiva y seleccionar el modelo más adecuado para identificar clientes con riesgo de abandono.

In [110]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [111]:
results = []

for model_name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.735608,0.501684,0.796791,0.615702
1,Decision Tree,0.716418,0.472160,0.566845,0.515188
2,Bagging,0.771144,0.572626,0.548128,0.560109
3,Random Forest,0.776830,0.581081,0.574866,0.577957
4,AdaBoost,0.744847,0.513089,0.786096,0.620908
5,Gradient Boosting,0.768301,0.551948,0.681818,0.610048


Sort Results

In [112]:
results_df.sort_values(by="Recall", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.735608,0.501684,0.796791,0.615702
4,AdaBoost,0.744847,0.513089,0.786096,0.620908
5,Gradient Boosting,0.768301,0.551948,0.681818,0.610048
3,Random Forest,0.776830,0.581081,0.574866,0.577957
1,Decision Tree,0.716418,0.472160,0.566845,0.515188
2,Bagging,0.771144,0.572626,0.548128,0.560109


## Conclusiones

Tras comparar los diferentes modelos, Logistic Regression obtuvo el mejor Recall para la clase Churn.

Dado que el objetivo principal del proyecto es identificar la mayor cantidad posible de clientes con riesgo de abandono, el Recall se considera la métrica más relevante.

Por este motivo, Logistic Regression entrenado con SMOTE se selecciona como el modelo final en esta fase del proyecto.